In [ ]:
dbutils.widgets.dropdown(
    "run_fixture_tests",
    "false",
    ["false", "true"]
)

run_fixture_tests = (
    dbutils.widgets.get("run_fixture_tests") == "true"
)

In [ ]:
dbutils.widgets.text("catalog" , "retail_operations_dev")
dbutils.widgets.text("source_schema" , "bronze")
dbutils.widgets.text("source_table" , "orders_raw")

In [ ]:
import re
from pyspark.sql import functions as F
from pyspark.sql import types as T

catalog = dbutils.widgets.get('catalog').strip()
source_schema = dbutils.widgets.get('source_schema').strip()
source_table = dbutils.widgets.get('source_table').strip()

for name in [catalog, source_schema, source_table]:
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", name):
        raise ValueError(f"Invalid identifier: {name}")

spark.conf.set("spark.sql.session.timeZone", "UTC")

source_name = f"{catalog}.{source_schema}.{source_table}"

raw_events = spark.table(source_name)

In [ ]:
item_schema = T.StructType([
    T.StructField('line_id' , T.IntegerType(), True),
    T.StructField('product_id' , T.StringType(), True),
    T.StructField('quantity' , T.IntegerType(), True),
    T.StructField('unit_price' , T.DecimalType(12,2), True)
])

event_schema = T.StructType([
    T.StructField("event_id" , T.StringType(), True),
    T.StructField('event_type' , T.StringType(), True),
    T.StructField('schema_version' , T.IntegerType(), True),
    T.StructField('order_id' , T.StringType(), True),    
    T.StructField('order_version' , T.LongType(), True),
    T.StructField('customer_id' , T.StringType(), True),
    T.StructField('event_time' , T.StringType(), True),
    T.StructField('status' , T.StringType(), True),
    T.StructField('currency' , T.StringType(), True),
    T.StructField('items' , T.ArrayType(item_schema), True),
    T.StructField('_corrupt_record' , T.StringType(), True),
])

In [ ]:
decoded_events = raw_events.select(
    F.col('key').cast("string").alias('raw_key'),
    F.col("value").cast('string').alias('raw_json')
)

parsed_events = (
    decoded_events
    .withColumn(
        "message_key",
        F.get_json_object('raw_key' , '$')
    )
    .withColumn(
        'event',
        F.from_json(
            F.col("raw_json"),
            event_schema,
            {
                "mode": "PERMISSIVE",
                "columnNameOfCorruptRecord": "_corrupt_record"
            }   
        )
    )
    .select(
        "raw_key",
        "raw_json",
        "message_key",
        "event.*"
    )
    .withColumn(
        "event_timestamp",
        F.try_to_timestamp("event_time")
    )
)


display(
    parsed_events
    .select(
        "event_id",
        "message_key",
        "order_id",
        "order_version",
        "event_timestamp",
        "status",
        "items",
        "_corrupt_record"
    )
)

In [ ]:
# A required string must contain something besides spaces.

def has_text(column):
    return F.coalesce(
        F.length(F.trim(column)) > 0,
        F.lit(False)
    )

# Check whether an order contains an invalid item.

invalid_item = F.exists(
    F.col('items'),
    lambda item : (
        item.isNull()
        | ~has_text(item["product_id"])
        | item["line_id"].isNull()
        | (item["line_id"] <= 0)
        | item["quantity"].isNull()
        | (item["quantity"] <= 0)
        | item["unit_price"].isNull()
        | (item["unit_price"] <= 0)
    )
)


# Extract line IDs so we can check for duplicates within an order.

line_ids = F.transform(
    F.col("items"),
    lambda item : item['line_id']
)

# Each pair contains a condition that must pass and its failure reason.

rules = [
    (
        has_text(F.col('raw_json'))
        & F.col("_corrupt_record").isNull(),
        "Malformed or missing event JSON"
    ),
    (has_text(F.col('event_id')), "Missing event_id"),
    (has_text(F.col('order_id')), "Missing order_id"),
    (has_text(F.col('customer_id')), "Missing customer_id"),
    (
        has_text(F.col("message_key"))
        & (F.col("message_key") == F.col("order_id")),
        'Kafka key missing or different from order_id'
    ),
    (
        F.col('schema_version') == 1,
        "Missing or unsupported schema_version"
    ),
    (
        F.col("order_version") > 0,
        "Missing or invalid order_version"
    ),
    (
        F.col("event_timestamp").isNotNull(),
        "Missing or invalid event_time"
    ),
    (
        F.col('event_type').isin(
            "order_created" , "order_updated" , "order_cancelled"
        ),
         "Missing or unsupported event_type"
    ),
    (
        F.col("status").isin(
            "placed", "paid", "shipped", "completed", "cancelled"
        ),
        "Missing or unsupported status"
    ),
    (
        F.col('currency') == "GBP",
        "Missing or unsupported currency"
    ),
    (
        F.col("items").isNotNull() & (F.size('items') > 0),
        "Missing or empty items"
    ),
    (
        ~F.coalesce(invalid_item, F.lit(False)),
        "An item has missing fields or non-positive numeric values"
    ),
    (
        F.size(line_ids) == F.size(F.array_distinct(line_ids)),
        "Duplicate line_id within the order"
    ),
     # A creation event must start at version 1 with status placed.
    (
        (F.col("event_type") != "order_created")
        | (
            (F.col("order_version") == 1)
            & (F.col("status") == "placed")
        ),
        "order_created requires version 1 and status placed"
    ),

    # Updates and cancellations must be later versions.
    (
        ~F.col("event_type").isin(
            "order_updated",
            "order_cancelled"
        )
        | (F.col("order_version") > 1),
        "Updates and cancellations require order_version greater than 1"
    ),

    # Cancellation event and cancellation status must agree.
    (
        (F.col("event_type") != "order_cancelled")
        | (F.col("status") == "cancelled"),
        "order_cancelled requires status cancelled"
    ),
    (
        (F.col("status") != "cancelled")
        | (F.col("event_type") == "order_cancelled"),
        "Status cancelled requires event_type order_cancelled"
    ),
]


# 5. Collect all failed rules for each event.

checked_events = (
    parsed_events
    .withColumn(
        'rejection_reasons',
        F.filter(
            F.array(
                *[
                    F.when(
                        ~F.coalesce(condition, F.lit(False)),
                        F.lit(reason)
                    )
                    for condition , reason in rules
                ]
            ),
            lambda reason : reason.isNotNull()
        )
    )
    .withColumn(
        "is_valid",
        (F.size('rejection_reasons') == 0)
    )
)

display(checked_events)

In [ ]:
# Separate events using the validation flag.
valid_events = checked_events.filter(
    F.col("is_valid")
)

quarantine_events = checked_events.filter(
    ~F.col("is_valid")
)

# Show the number of events in each category.
display(
    checked_events
    .groupBy("is_valid")
    .count()
    .orderBy("is_valid")
)

# Show rejected events and their reasons.
display(
    quarantine_events.select(
        "event_id",
        "order_id",
        "rejection_reasons",
        "raw_json"
    )
)

In [ ]:
# 1. Select one valid event as the starting point.

baseline_event = valid_events.limit(1)

if baseline_event.count() == 0 :
    raise ValueError('The test requires at least one valid event.')


# 2. Create test cases with known expected outcomes.

test_cases = [
    (
        baseline_event,
        "valid_event",
        True,
        None
    ),
    (
        baseline_event
        .withColumn(
            "customer_id",
            F.lit(None).cast('string')
        ),
        "missing_customer",
        False,
        "Missing customer_id"
    ),
    (
        baseline_event
        .withColumn(
            "currency",
            F.lit("USD")
        ),
        "unsupported_currency",
        False,
        "Missing or unsupported currency"
    ),
    (
        baseline_event
        .withColumn("event_type" , F.lit('order_cancelled'))
        .withColumn("order_version" , F.lit(2).cast("long"))
        .withColumn("status" , F.lit("paid")),
        "cancellation_with_paid_status",
        False,
        "order_cancelled requires status cancelled"
    ),
    (
        baseline_event
        .withColumn(
            "items",
            F.concat(
                F.slice(F.col("items") , 1, 1),
                F.slice(F.col("items") , 1, 1),
            )
        ),
        "duplicate_line_id",
        False,
        "Duplicate line_id within the order"
    )
]


# 3. Combine the test cases into one DataFrame.

test_frames = [
    frame
    .withColumn('test_name' , F.lit(name))
    .withColumn('expected_valid', F.lit(expected_valid))
    .withColumn(
        "expected_reason",
        F.lit(expected_reason).cast('string')
    )
    for frame , name , expected_valid , expected_reason in test_cases
]

test_events = test_frames[0]

for frame in test_frames[1:]:
    test_events = test_events.unionByName(frame)


# 4. Recalculate validation using the rules from Cell 4.
test_results = (
    test_events
    .withColumn(
        "rejection_reasons",
        F.filter(
            F.array(*[
                F.when(
                    ~F.coalesce(condition, F.lit(False)),
                    F.lit(reason)
                )
                for condition, reason in rules
            ]),
            lambda reason: reason.isNotNull()
        )
    )
    .withColumn(
        "is_valid",
        F.size("rejection_reasons") == 0
    )
    .withColumn(
        "test_passed",
        (F.col("is_valid") == F.col("expected_valid"))
        & (
            F.col("expected_reason").isNull()
            | F.array_contains(
                F.col("rejection_reasons"),
                F.col("expected_reason")
            )
        )
    )
)

display(
    test_results.select(
        "test_name",
        "expected_valid",
        "is_valid",
        "rejection_reasons",
        "expected_reason",
        "test_passed"
    ).orderBy("test_name")
)

failed_tests = (
    test_results
    .filter(~F.coalesce(F.col("test_passed"), F.lit("false")))
).count()

assert failed_tests == 0, f"{failed_tests} validation tests failed."

In [ ]:
def parse_and_validate_orders(raw_df):
    # Convert Kafka bytes into readable JSON text.
    decoded_df = raw_df.select(
        F.col("key").cast("string").alias("raw_key"),
        F.col("value").cast("string").alias("raw_json"),
    )

    # Extract the key and parse the event fields.
    parsed_df = (
        decoded_df
        .withColumn(
            "message_key",
            F.get_json_object("raw_key", "$")
        )
        .withColumn(
            "event",
            F.from_json(
                F.col("raw_json"),
                event_schema,
                {
                    "mode": "PERMISSIVE",
                    "columnNameOfCorruptRecord": "_corrupt_record",
                },
            )
        )
        .select(
            "raw_key",
            "raw_json",
            "message_key",
            "event.*",
        )
        .withColumn(
            "event_timestamp",
            F.try_to_timestamp("event_time")
        )
    )

    # Apply the validation rules defined in Cell 4.
    checked_df = (
        parsed_df
        .withColumn(
            "rejection_reasons",
            F.filter(
                F.array(*[
                    F.when(
                        ~F.coalesce(condition, F.lit(False)),
                        F.lit(reason)
                    )
                    for condition, reason in rules
                ]),
                lambda reason: reason.isNotNull()
            )
        )
        .withColumn(
            "is_valid",
            F.size("rejection_reasons") == 0
        )
    )

    return checked_df

In [ ]:
# Read the Bronze Delta table as a streaming source.

raw_order_stream = (
    spark.readStream
    .table(source_name)
)

checked_order_stream = parse_and_validate_orders(raw_order_stream)

print("Streaming Dataframe : " , checked_order_stream.isStreaming)

checked_order_stream.printSchema()

In [ ]:
# Prepare the operations schema and a volume for checkpoints.
spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {catalog}.operations"
)

spark.sql(
    f"CREATE VOLUME IF NOT EXISTS "
    f"{catalog}.operations.streaming_checkpoints"
)

# Define the destination table.
validated_table = f"{catalog}.operations.order_events_validated"

# Give this streaming query a permanent checkpoint directory.
checkpoint_path = (
    f"/Volumes/{catalog}/operations/streaming_checkpoints/"
    "bronze_to_validated_orders_v1"
)
print("Destination table:", validated_table)
print("Checkpoint directory:", checkpoint_path)

In [ ]:
# Add the time at which our validation stream processes each event.

events_to_write = (
    checked_order_stream
    .withColumn(
        "validated_at",
        F.current_timestamp()
    )
)

# Start an incremental run that finishes after processing available data.
validation_query = (
    events_to_write
    .writeStream
    .format("delta")
    .outputMode('append')
    .option('checkpointLocation' , checkpoint_path)
    .trigger(availableNow = True)
    .toTable(validated_table)
)

# Wait for the streaming run to finish.

validation_query.awaitTermination()

print(f"Streaming run completed: {validated_table}")

In [ ]:
saved_events = spark.table(validated_table)

display(
    saved_events
    .groupBy("is_valid")
    .count()
    .orderBy("is_valid")
)

display(
    saved_events
    .select(
        "event_id",
        "order_id",
        "is_valid",
        "rejection_reasons",
        "validated_at",
    )
    .limit(10)
)

In [ ]:
# Ensure the Silver schema exists.
spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {catalog}.silver"
)

# Define the view names.

valid_candidates_view =  (f"{catalog}.silver.order_events_valid_candidates")
quarantine_view = (f"{catalog}.silver.order_events_quarantine")


# Expose events that passed field and item validation.

spark.sql(f"""
    CREATE OR REPLACE VIEW {valid_candidates_view} AS
    SELECT *
    From {validated_table}
    where is_valid = TRUE
""")

# Expose events that failed validation.
spark.sql(f"""
    CREATE OR REPLACE VIEW {quarantine_view} AS
    SELECT *
    From {validated_table}  
    where is_valid = FALSE
""")

print("Valid candidates:", valid_candidates_view)
print("Quarantine:", quarantine_view)


In [ ]:
# Read the saved table and its two views.

validated_count = spark.table(validated_table).count()

valid_count = spark.table(
    valid_candidates_view
).count()

quarantine_count = spark.table(
    quarantine_view
).count()

print(f"Validated records: {validated_count}")
print(f"Valid candidates: {valid_count}")
print(f"Quarantined records: {quarantine_count}")

# Every saved event should appear in exactly one category.
assert validated_count == valid_count + quarantine_count, (
    "Some records are not classified as valid or invalid."
)

# Inspect any rejected events.
display(
    spark.table(quarantine_view).select(
        "event_id",
        "order_id",
        "rejection_reasons",
        "raw_json",
    )
)

In [ ]:
# Read events that passed validation.

candidate_events = spark.table(valid_candidates_view)

# Define the business fields used to compare events.

event_fields = [
    "event_id",
    "event_type",
    "schema_version",
    "order_id",
    "order_version",
    "customer_id",
    "event_time",
    "status",
    "currency",
    "items",
]

# Serialize the parsed business fields in a consistent order.

fingerprinted_events = (
    candidate_events
    .withColumn(
        "event_payload",
        F.to_json(
            F.struct(*[
                F.col(field_name)
                for field_name in event_fields
            ]),
            options={"ignoreNullFields" : "false"}
        )
    )
)

display(
    fingerprinted_events.select(
        "event_id",
        "order_id",
        "event_payload",
    ).limit(5)
)



In [ ]:
# Summarize all deliveries of each event ID.
event_id_checks = (
    fingerprinted_events
    .groupby("event_id")
    .agg(
        F.count("*").alias("delivery_count"),
        F.countDistinct("event_payload").alias("payload_versions")
    )
    .withColumn(
        "classification",
        F.when(
            (F.col("payload_versions") > 1),
            F.lit("conflict")
        )
        .when(
            (F.col("delivery_count") > 1),
            F.lit("duplicate")
        )
        .otherwise(
            F.lit("unique")
        )
    )
)


# Summarize the classifications.

display(
    event_id_checks
    .groupBy("classification")
    .count()
)

# Inspect duplicates or conflicts, if any.
display(
    event_id_checks
    .filter(F.col("classification") != "unique")
    .orderBy("event_id")
)

In [ ]:
# We’ll create two DataFrames:
#- deduplicated_events: one record per non-conflicting event ID.
#- conflicting_events: all records whose event ID has multiple payloads.

from pyspark.sql.window import Window

# Attach each event ID's classification to its records.

classified_events = fingerprinted_events.join(
    event_id_checks.select(
        "event_id",
        "delivery_count",
        "payload_versions",
        "classification"
    ),
    on = "event_id",
    how = 'inner'
)

# Preserve every record associated with a conflicting event ID.

conflicting_events = (
    classified_events
    .filter(
        F.col("classification") == "conflict"
    )
)

# Keep unique events and identical duplicate deliveries.

non_conflicting_events = (
    classified_events
    .filter(
        F.col("classification") != "conflict"
    )
)

# Decide which delivery to retain for each event ID.

delivery_window = Window.partitionBy("event_id").orderBy(
    F.col("validated_at").asc(),
    F.col("raw_json").asc(),
    F.col("raw_key").asc()
)

# Keep the first delivery of each non-conflicting event.

deduplicated_events = (
    non_conflicting_events
    .withColumn(
        "delivery_rank",
        F.row_number().over(delivery_window)
    )
    .filter(
        F.col("delivery_rank") == 1
    )
    .drop('delivery_rank')
)


display(
    deduplicated_events.select(
        "event_id",
        "order_id",
        "order_version",
        "delivery_count",
        "classification",
    ).limit(10)
)

In [ ]:
# Count the output records and distinct event IDs together.

dedup_summary = (
    deduplicated_events
    .agg(
        F.count("*").alias("event_count"),
        F.countDistinct("event_id").alias("unique_event_ids")
    )
)
display(dedup_summary)

# Check that there is exactly one output row per event ID.
summary = dedup_summary.first()

assert summary["event_count"] == summary["unique_event_ids"],(
    "Duplicate event IDs remain in the output"
)

# Every non-conflicting ID should have one retained record.
expected_event_count = (
    event_id_checks
    .filter(
        F.col("classification") != "conflict"
    ).count()
)

assert summary["event_count"] == expected_event_count ,(
    "Some non-conflicting event IDs are missing."
)

# Inspect conflicting records separately.
display(
    conflicting_events.select(
        "event_id",
        "order_id",
        "payload_versions",
        "event_payload",
        "raw_json",
    )
)

In [ ]:
# Stop before changing Silver if an event ID has different payloads.
if conflicting_events.limit(1).count() > 0:
    display(conflicting_events)

    raise ValueError(
        "Conflicting order event IDs detected. "
        "Silver orders were not updated by this run. "
        "Review the conflicting records before retrying."
    )
    
from delta.tables import DeltaTable

silver_events_table = f"{catalog}.silver.order_events"

# Create an empty Delta table with the required schema if needed.

if not spark.catalog.tableExists(silver_events_table):
    (
        deduplicated_events
        .limit(0)
        .write
        .format('delta')
        .mode("overwrite")
        .saveAsTable(silver_events_table)
    )


# Synchronize the table with the complete deduplicated dataset.

(
    DeltaTable.forName(spark, silver_events_table)
    .alias('target')
    .merge(
        deduplicated_events.alias("source"),
        "target.event_id = source.event_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .whenNotMatchedBySourceDelete()
    .execute()
)

print(f"Saved Silver events to: {silver_events_table}")

In [ ]:
saved_silver_events = spark.table(silver_events_table)

summary_df = saved_silver_events.agg(
    F.count("*").alias("saved_events"),
    F.countDistinct("event_id").alias('unique_event_ids'),
    F.countDistinct("order_id").alias("unique_orders"),
)


display(summary_df)

summary = summary_df.first()

assert summary["saved_events"] == summary["unique_event_ids"],(
    "The Silver table contains duplicate event IDs."
)

assert summary["saved_events"] == deduplicated_events.count(),(
    f"Expected {deduplicated_events.count()} events, "
    f"but found {summary['saved_events']}."
)

display(
    saved_silver_events.select(
        "event_id",
        "order_id",
        "order_version",
        "status",
        "delivery_count",
    ).limit(10)
)

In [ ]:
from pyspark.sql.window import Window

# Read the saved event history.

order_history = spark.table(f"{catalog}.silver.order_events")

# Check for different events claiming the same order version.

version_collisions = (
    order_history
    .groupBy("order_id", "order_version")
    .count()
    .filter(F.col("count") > 1)
)

if version_collisions.limit(1).count() > 0 :
    raise ValueError(
        "Multiple events claim the same order version. "
        "Resolve these before building current orders."
    )

# Rank versions from highest to lowest within each order.

order_window = Window.partitionBy("order_id").orderBy(F.col('order_version').desc())


# Keep the highest accepted version.

current_orders = (
    order_history
    .withColumn(
        "version_rank",
        F.row_number().over(order_window)
    )
    .filter(F.col("version_rank") == 1)
    .select(
        "order_id",
        "customer_id",
        "order_version",
        F.col("event_id").alias("latest_event_id"),
        "event_timestamp",
        "status",
        "currency",
        "items",
    )
)

display(
    current_orders.filter(
        F.col("order_id") == "O10001"
    )
)

In [ ]:
from delta.tables import DeltaTable

current_orders_table = f"{catalog}.silver.orders_current"

# Make the DataFrame available to SQL.

current_orders.createOrReplaceTempView(
    "current_orders_source"
)

# Create the empty destination if it does not exist.

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {current_orders_table} 
    USING DELTA
    AS
    SELECT *
    FROM current_orders_source
    where 1 = 0        
""")

# Synchronize with the complete current-order dataset.

(
    DeltaTable.forName(spark , current_orders_table)
    .alias('target')
    .merge(
        current_orders.alias("source"),
        "target.order_id = source.order_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .whenNotMatchedBySourceDelete()
    .execute()
)

print(f"Saved: {current_orders_table}")

In [ ]:
# Read the saved current-order snapshots.
current_order_snapshot = spark.table(
    f"{catalog}.silver.orders_current"
)

# Turn each item in an order's array into a separate row.
current_items = (
    current_order_snapshot
    .select(
        "order_id",
        "order_version",
        "latest_event_id",
        "customer_id",
        "status",
        "currency",
        F.explode("items").alias("item"),
    )
    .select(
        "order_id",
        "order_version",
        "latest_event_id",
        "customer_id",
        F.col("status").alias("order_status"),
        "currency",
        F.col("item.line_id").alias("line_id"),
        F.col("item.product_id").alias("product_id"),
        F.col("item.quantity").alias("quantity"),
        F.col("item.unit_price").alias("unit_price")
    )
    .withColumn(
        "line_amount",
        (
            F.col("quantity") * F.col("unit_price")
        ).cast("decimal(23,2)")
    )
)

display(
    current_items
    .filter(F.col("order_id") == "O10001")
)

In [ ]:
from delta.tables import DeltaTable

current_items_table = (
    f"{catalog}.silver.order_items_current"
)

# Verify that each order-line combination is unique.
duplicate_item_keys = (
    current_items
    .groupBy("order_id", "line_id")
    .count()
    .filter(F.col('count') > 1)
)

if duplicate_item_keys.limit(1).count() > 0 :
    raise ValueError("Duplicate order_id and line_id combinations.")

# Expose the source to SQL.

current_items.createOrReplaceTempView(
    "current_order_items_source"
)

# Create an empty destination table if needed.

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {current_items_table}
    USING delta
    AS
    SELECT * 
    FROM current_order_items_source
    WHERE 1 = 0
""")

# Synchronize the complete current-item dataset.
(
    DeltaTable.forName(spark , current_items_table)
    .alias("target")
    .merge(
        current_items.alias("source"),
        """
        target.order_id = source.order_id
        AND
        target.line_id = source.line_id
        """
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .whenNotMatchedBySourceDelete()
    .execute()
)

print(f"Saved: {current_items_table}")

In [ ]:
saved_items = spark.table(current_items_table)

# Each element in a current order's items array should become one row.
expected_item_count = (
    current_order_snapshot
    .agg(
        F.coalesce(
            F.sum(F.size("items")),
            F.lit(0)
        ).alias("expected_items")
    )
    .first()["expected_items"]
)

actual_item_count = saved_items.count()

unique_item_count = (
    saved_items
    .select("order_id", "line_id")
    .distinct()
    .count()
)

print("Expected items:", expected_item_count)
print("Saved items:", actual_item_count)
print("Unique order-line keys:", unique_item_count)

assert actual_item_count == expected_item_count, (
    "The saved item count differs from the current-order snapshots."
)

assert actual_item_count == unique_item_count, (
    "Duplicate order-line keys exist in the saved table."
)

display(
    saved_items
    .filter(F.col("order_id") == "O10001")
    .orderBy("line_id")
)

print("Current-item checks passed.")